# Fed CP 来源传输限定修复：终端审计

只读四次已消耗请求和NAS失败收据，不再访问外部来源。

In [1]:
import hashlib,json
from pathlib import Path
from src.data.minute_intake import atomic_json
root=Path.cwd();card=root/'research/fed_cp_rating_spread_source_20261023'
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
base=json.loads((card/'plan.json').read_text());repair=json.loads((card/'transport_repair_1_plan.json').read_text());first=json.loads((card/'retrieval_receipt.json').read_text());second=json.loads((card/'transport_repair_1_retrieval.json').read_text());failure=json.loads((card/'transport_repair_1_failure.json').read_text());archive=json.loads((card/'source_archive_receipt.json').read_text())
assert sha(card/'plan.json')==repair['original_plan_sha256'] and sha(card/'retrieval_receipt.json')==repair['original_retrieval_receipt_sha256'] and sha(card/'source_archive_receipt.json')==repair['original_source_archive_receipt_sha256']
assert sha(card/'transport_repair_1_plan.json')==second['repair_plan_sha256']==failure['repair_plan_sha256']
assert [x['curl_exit_code'] for x in first['responses']]==[92,92] and [x['curl_exit_code'] for x in second['responses']]==[28,28]
assert all(x['body_bytes']==x['headers_bytes']==0 and x['http_status']==0 for x in first['responses']+second['responses'])
assert archive['status']=='exact_received_bytes_NAS_scoped_restore_verified' and archive['scoped_members_verified']==4
assert failure['source_data_bytes_received']==failure['source_values_parsed']==failure['Qlib_D_calendar_calls']==failure['model_fits']==failure['native_accounts']==0
for folder in [root/'data/snapshots/fed_cp_rating_spread_source_20261023',root/'data/snapshots/fed_cp_rating_spread_source_20261023/repair1']:
 for req in base['requests']:
  for suffix in ('.csv','.headers.txt'):
   f=folder/(req['series']+suffix);assert f.is_file() and f.stat().st_size==0
state=json.loads((card/'repair1_run_state.json').read_text());state.update(status='HTTP1_1_transport_terminal_no_source',Qlib_D_calendar_calls=0,new_Qlib_Recorders=0);atomic_json(state,card/'repair1_run_state.json')
summary={'experiment':base['experiment'],'decision':'needs_original_Fed_CP_transport_after_two_bounded_protocols_no_fit','objective_achieved':False,'original_plan_sha256':sha(card/'plan.json'),'original_retrieval_sha256':sha(card/'retrieval_receipt.json'),'original_source_NAS_sha256':archive['archive_sha256'],'repair_plan_sha256':sha(card/'transport_repair_1_plan.json'),'repair_retrieval_sha256':sha(card/'transport_repair_1_retrieval.json'),'repair_failure':'transport_repair_1_failure.json','cumulative_HTTP_GET_attempts':4,'source_data_bytes_received':0,'source_first_public_vintage_qualified':False,'Qlib_D_calendar_calls':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
atomic_json(summary,card/'summary.json')
(card/'conclusion.md').write_text('# Fed商业票据等级利差：两次定额传输均未取得原文\n\n拟议信号是联储公开的30日非金融A2/P2与AA票据利率差，不同于已失败的SOFR−IORB和HYG−LQD。联储[CP说明](https://www.federalreserve.gov/releases/cp/about.htm)描述同日、同期限利率来源与通常次日发布，但不保证历史每次实际钟点；[FRED A2/P2](https://fred.stlouisfed.org/series/RIFSPPNA2P2D30NB/)和[FRED AA](https://fred.stlouisfed.org/series/RIFSPPNAAD30NB)是当前回溯入口，不是原始发布版本。\n\n冻结的两次默认传输均以curl 92结束，0 HTTP响应/报头/正文；其空响应字节及收据已归档NAS并限定恢复。另冻的同URL HTTP/1.1窄修复各请求一次，均以curl 28、约30秒无字节结束。修复的NAS归档PUT也以curl 28超时，随后只读SSH路径校验失败；这四份本地空响应保留，**未证明**修复归档落盘。两次修复都在来源解析前停止，累计4次请求、0条数值、0 Qlib映射、0拟合、0账户。\n\n决定：`needs_original_Fed_CP_transport_after_two_bounded_protocols_no_fit`。这是本机传输和归档状态，不是票据利差信息门失败，也不能据FRED网页的今天数值回填2011–2022。固定传输版本关闭，不继续改端点、协议或日期救援；需独立授权的新有限来源路径或外部网络状态变化。SOXX总Goal继续走其他不依赖该来源的支线。\n',encoding='utf-8')
print(summary['decision'],summary['cumulative_HTTP_GET_attempts'],summary['source_data_bytes_received'])

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


needs_original_Fed_CP_transport_after_two_bounded_protocols_no_fit 4 0


## 中文结论

见 conclusion.md；来源未取得，0拟合、0账户。